In [1]:
import pandas as pd
import numpy as np

In [91]:
quali_2025 = pd.read_csv('race_quali_2025.csv')
quali_2024 = pd.read_csv('race_quali_2024.csv')
quali_2023 = pd.read_csv('race_quali_2023.csv')
quali_2022 = pd.read_csv('race_quali_2022.csv')

In [95]:
quali_2022 = quali_2022.drop('raceId',axis = 1)
quali_2023 = quali_2023.drop('raceId',axis = 1)
quali_2024 = quali_2024.drop('raceId',axis = 1)
quali_2025 = quali_2025.drop('raceId',axis = 1)

In [97]:
quali_2022.to_csv("quali-dataset/race_quali_2022.csv", index=False)
quali_2023.to_csv("quali-dataset/race_quali_2023.csv", index=False)
quali_2024.to_csv("quali-dataset/race_quali_2024.csv", index=False)
quali_2025.to_csv("quali-dataset/race_quali_2025.csv", index=False)

In [96]:
quali_2025.head()

,Track,Position,No,Driver,Team,Q1,Q2,Q3,Laps,kaggle_driver_id
0,Australia,1,4,Lando Norris,McLaren Mercedes,1:15.912,1:15.415,1:15.096,20,846
1,Australia,2,81,Oscar Piastri,McLaren Mercedes,1:16.062,1:15.468,1:15.180,18,857
2,Australia,3,1,Max Verstappen,Red Bull Racing Honda RBPT,1:16.018,1:15.565,1:15.481,17,830
3,Australia,4,63,George Russell,Mercedes,1:15.971,1:15.798,1:15.546,21,847
4,Australia,5,22,Yuki Tsunoda,Racing Bulls Honda RBPT,1:16.225,1:16.009,1:15.670,18,852


In [98]:
sprint_quali_2021 = pd.read_csv('sprint_quali_2021.csv')
sprint_quali_2023 = pd.read_csv('sprint_quali_2023.csv')
sprint_quali_2024 = pd.read_csv('sprint_quali_2024.csv')

In [99]:
sprint_quali_2024.head()

,Track,Position,No,Driver,Team,Q1,Q2,Q3,Laps,kaggle_driver_id
0,China,1,4,Lando Norris,McLaren Mercedes,1:36.384,1:36.047,1:57.940,13,846
1,China,2,44,Lewis Hamilton,Mercedes,1:37.181,1:36.287,1:59.201,15,1
2,China,3,14,Fernando Alonso,Aston Martin Aramco Mercedes,1:36.883,1:36.119,1:59.915,14,4
3,China,4,1,Max Verstappen,Red Bull Racing Honda RBPT,1:36.456,1:35.606,2:00.028,12,830
4,China,5,55,Carlos Sainz,Ferrari,1:36.719,1:36.052,2:00.214,15,832


In [55]:
kaggle_quali = pd.read_csv('filler-datasets/qualifying.csv')
kaggle_quali_from_2022 = kaggle_quali[kaggle_quali['raceId']>1073]

In [57]:
kaggle_race_ids_from_2022 = (
    kaggle_quali_from_2022[['raceId']]
    .drop_duplicates()
    .reset_index(drop=True)['raceId']
    .tolist()
)

In [35]:
kaggle_sprint_res = pd.read_csv('filler-datasets/kaggle_sprint_res.csv')
kaggle_race = pd.read_csv('filler-datasets/kaggle_race.csv')
kaggle_driver = pd.read_csv('filler-datasets/drivers_filtered.csv')
kaggle_circuits = pd.read_csv('filler-datasets/circuits.csv')

In [43]:
quali_2023.head()

,raceId,Track,Team,kaggle_driver_id,Position,participated_q2,participated_q3,q1_time_sec,q2_time_sec,q3_time_sec
0,1074,Bahrain,Ferrari,844,1,1,1,91.471,90.932,90.558
1,1074,Bahrain,Red Bull Racing RBPT,830,2,1,1,91.785,90.757,90.681
2,1074,Bahrain,Ferrari,832,3,1,1,91.567,90.787,90.687
3,1074,Bahrain,Red Bull Racing RBPT,815,4,1,1,92.311,91.008,90.921
4,1074,Bahrain,Mercedes,1,5,1,1,92.285,91.048,91.238


In [26]:
def quali_time_to_sec(val):
    if pd.isna(val):
        return np.nan

    val = str(val).strip()
    parts = val.split(':')

    try:
        if len(parts) == 2:
            # mm:ss.xxx
            m, s = parts
            return int(m) * 60 + float(s)
        elif len(parts) == 3:
            # hh:mm:ss.xxx (rare but exists)
            h, m, s = parts
            return int(h) * 3600 + int(m) * 60 + float(s)
    except:
        return np.nan

    return np.nan
def process_quali(df):
    df = df.copy()

    # participation flags
    df['participated_q2'] = df['Q2'].notna().astype(int)
    df['participated_q3'] = df['Q3'].notna().astype(int)

    # parse times
    df['q1_time_sec'] = df['Q1'].apply(quali_time_to_sec)
    df['q2_time_sec'] = df['Q2'].apply(quali_time_to_sec)
    df['q3_time_sec'] = df['Q3'].apply(quali_time_to_sec)

    # keep final columns
    df = df[
        [
            'raceId',
            'Track',
            'Team',
            'kaggle_driver_id',
            'Position',
            'participated_q2',
            'participated_q3',
            'q1_time_sec',
            'q2_time_sec',
            'q3_time_sec',
        ]
    ]

    return df


In [27]:
def normalize_name_series(s):
    return (
        s.str.lower()
         .str.strip()
         .str.normalize('NFKD')
         .str.encode('ascii', errors='ignore')
         .str.decode('utf-8')
    )

def add_kaggle_driver_id(df, kaggle_driver):
    df = df.copy()

    df['driver_norm'] = normalize_name_series(df['Driver'])

    kd = kaggle_driver.copy()
    kd['driver_norm'] = normalize_name_series(
        kd['forename'] + ' ' + kd['surname']
    )

    df = df.merge(
        kd[['driverId', 'driver_norm']],
        on='driver_norm',
        how='left'
    )

    df.rename(columns={'driverId': 'kaggle_driver_id'}, inplace=True)
    df.drop(columns=['driver_norm'], inplace=True)

    return df


In [47]:
quali_2022 = add_kaggle_driver_id(quali_2022, kaggle_driver)
#quali_2023 = add_kaggle_driver_id(quali_2023, kaggle_driver)
#quali_2024 = add_kaggle_driver_id(quali_2024, kaggle_driver)
#quali_2025 = add_kaggle_driver_id(quali_2025, kaggle_driver)

In [52]:
quali_2022 = process_quali(quali_2022)
quali_2023 = process_quali(quali_2023)
quali_2024 = process_quali(quali_2024)
quali_2025 = process_quali(quali_2025)

In [72]:
sprint_quali_2021.head()

,raceId,Track,Team,kaggle_driver_id,Position,participated_q2,participated_q3,q1_time_sec,q2_time_sec,q3_time_sec
0,1074,China,Ferrari,1,1,1,1,91.212,91.484,90.849
1,1074,China,Red Bull Racing Honda RBPT,830,2,1,1,91.916,91.521,90.867
2,1074,China,McLaren Mercedes,857,3,1,1,91.723,91.362,90.929
3,1074,China,Ferrari,844,4,1,1,91.518,91.561,91.057
4,1074,China,Mercedes,847,5,1,1,91.952,91.346,91.169


In [58]:
def assign_race_id_by_track(df, race_ids):
    df = df.copy()

    race_id_col = []
    race_ptr = 0

    tracks = df['Track'].values
    n = len(df)

    start = 0
    for i in range(1, n):
        if tracks[i] != tracks[i - 1]:
            # assign raceId to previous block
            race_id_col.extend([race_ids[race_ptr]] * (i - start))
            race_ptr += 1
            start = i

    # last block
    race_id_col.extend([race_ids[race_ptr]] * (n - start))
    race_ptr += 1

    # safety
    assert len(race_id_col) == len(df), "raceId length mismatch"

    df['raceId'] = race_id_col
    return df
# race qualifying
sprint_quali_2021 = assign_race_id_by_track(sprint_quali_2021, kaggle_race_ids_from_2022)
sprint_quali_2023 = assign_race_id_by_track(sprint_quali_2023, kaggle_race_ids_from_2022)
sprint_quali_2024 = assign_race_id_by_track(sprint_quali_2024, kaggle_race_ids_from_2022)

In [84]:
quali_2023['raceId'].value_counts()

raceId
1074    20
1075    20
1076    20
1077    20
1078    20
1079    20
1080    20
1081    20
1082    20
1083    20
1084    20
1085    20
1086    20
1087    20
1088    20
1089    20
1091    20
1092    20
1093    20
1094    20
1095    20
1096    20
Name: count, dtype: int64

In [70]:
import numpy as np

cols = ['SQ1', 'SQ2', 'SQ3']

existing_cols = sprint_quali_2021.columns.intersection(cols)

sprint_quali_2021[existing_cols] = (
    sprint_quali_2021[existing_cols]
    .replace('DNF', np.nan)
)

In [63]:
def sprint_quali_time_to_sec(val):
    if pd.isna(val):
        return np.nan

    val = str(val).strip()
    parts = val.split(':')

    try:
        if len(parts) == 2:
            # mm:ss.xxx
            m, s = parts
            return int(m) * 60 + float(s)
        elif len(parts) == 3:
            # hh:mm:ss.xxx (rare but exists)
            h, m, s = parts
            return int(h) * 3600 + int(m) * 60 + float(s)
    except:
        return np.nan

    return np.nan
def sprint_process_quali(df):
    df = df.copy()

    # participation flags
    df['participated_q2'] = df['SQ2'].notna().astype(int)
    df['participated_q3'] = df['SQ3'].notna().astype(int)

    # parse times
    df['q1_time_sec'] = df['SQ1'].apply(sprint_quali_time_to_sec)
    df['q2_time_sec'] = df['SQ2'].apply(sprint_quali_time_to_sec)
    df['q3_time_sec'] = df['SQ3'].apply(sprint_quali_time_to_sec)

    # keep final columns
    df = df[
        [
            'raceId',
            'Track',
            'Team',
            'kaggle_driver_id',
            'Position',
            'participated_q2',
            'participated_q3',
            'q1_time_sec',
            'q2_time_sec',
            'q3_time_sec',
        ]
    ]

    return df


In [77]:
#sprint_quali_2021 = process_quali(sprint_quali_2021)
#sprint_quali_2023 = sprint_process_quali(sprint_quali_2023)
sprint_quali_2024 = process_quali(sprint_quali_2024)

KeyError: 'SQ2'